# PC³ — quick demo

**Physics-Constrained Conformal Prediction for Composites.** This notebook fits PC³ on synthetic composite data with rigorous Voigt–Reuss bounds and checks two things:

1. the intervals attain the target **1−α** coverage, and
2. they never leave the physically admissible corridor **[L(x), U(x)]** (zero violations).

Run from the repository root so that `pc3.py` is importable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pc3

# synthetic composite data with rigorous Voigt-Reuss bounds
X, y, mono, bounds_fn, *_ = pc3.make_synthetic_composite(n=600, seed=0)
ntr, ncal = 300, 150
Xtr, ytr = X[:ntr], y[:ntr]
Xcal, ycal = X[ntr:ntr + ncal], y[ntr:ntr + ncal]
Xte, yte = X[ntr + ncal:], y[ntr + ncal:]

## Fit PC³

Monotone quantile base models → physics-aware conformal calibration (infinite score for out-of-corridor calibration points) → projection into `[L(x), U(x)]`.

In [ ]:
model = pc3.PC3(mono, bounds_fn, "cqr", True, True, True, alpha=0.1)
model.fit(Xtr, ytr).calibrate(Xcal, ycal)
pred, lo, hi, _, _ = model.predict(Xte)

cov = float(np.mean((yte >= lo) & (yte <= hi)))
L, U = bounds_fn(Xte)
viol = float(np.mean((lo < L - 1e-9) | (hi > U + 1e-9)))
print(f"empirical coverage : {cov*100:.1f}%  (target 90%)")
print(f"physical violations: {viol*100:.1f}%  (should be 0.0%)")
print(f"mean interval width: {np.mean(hi - lo):.2f}")

In [ ]:
order = np.argsort(yte)
plt.figure(figsize=(6, 4))
plt.fill_between(yte[order], lo[order], hi[order], alpha=0.3, label="90% interval")
plt.plot(yte[order], pred[order], ".", ms=4, label="prediction")
plt.plot(yte[order], yte[order], "k--", lw=1, label="ideal")
plt.xlabel("True value"); plt.ylabel("Prediction +/- interval")
plt.legend(); plt.tight_layout(); plt.show()

For the full set of experiments behind the paper, run `python run_all.py` from the repository root. See the [README](../README.md) for the script→figure mapping.